# tk-assistant · Эксперимент №4: QLoRA 6 эпох на датасете v2 (adapter_e6_v3rag)

**Runtime → Change runtime type → T4 GPU**, затем ячейки сверху вниз.

Датасет v2: ~3200 пар (две волны генерации). Обучение ~1,5–2 часа (2400 шагов).
Трекинг — TensorBoard, ключи не нужны. Конфиг: `configs/train_qlora_e6v3rag.yaml`.

⚠️ Запускать только после подтверждения, что v2-датасет собран (train_ragnative.jsonl перестроен).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install unsloth tensorboard
!unsloth install-kernels xformers

## 1. Свежий клон + чистое окружение

В выводе проверь: последний коммит репозитория и `report_to: tensorboard`.

In [ ]:
%cd /content
!rm -rf tk-assistant
!git clone https://github.com/markkentos/tk-assistant.git
%cd /content/tk-assistant
!git log --oneline -1
!grep report_to configs/train_qlora_e6v3rag.yaml
import os
os.environ.pop("WANDB_API_KEY", None)
print("окружение почищено")

In [ ]:
# Проверка датасета: *_ragnative должны существовать, в большинстве строк — candidates
import json
for name in ("train", "val"):
    rows = [json.loads(l) for l in open(f"data/processed/{name}_ragnative.jsonl", encoding="utf-8")]
    with_ctx = sum(1 for r in rows if r.get("candidates"))
    print(f"{name}: {len(rows)} пар, с контекстом: {with_ctx}")
    assert with_ctx > 0
    assert len(rows) > 2000 if name == "train" else True

## 2. Обучение: 4 эпохи (~1,5–2 часа, ~1600 шагов)

Следи за loss-строками. В конце — 10 пробных ответов: появились ли правильные
номера статей — главный признак успеха.

In [ ]:
!python src/train_qlora.py --config configs/train_qlora_e6v3rag.yaml

## 2.1 Графики обучения (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/adapter_e6_v3rag/checkpoints

## 3. Скачать адаптер

In [ ]:
!zip -r adapter_e6_v3rag.zip outputs/adapter_e6_v3rag -x "*/checkpoints/*"
from google.colab import files
files.download("adapter_e6_v3rag.zip")

## 4. Ответы на бенчмарке (той же сессией, ~15 минут)

Скачается `answers_tuned_e6v3rag.jsonl` — положи в `eval/runs/` проекта.

In [ ]:
!python src/generate_answers.py --mode tuned --adapter outputs/adapter_e6_v3rag --input eval/benchmark.jsonl --output eval/runs/answers_tuned_e6v3rag.jsonl
files.download("eval/runs/answers_tuned_e6v3rag.jsonl")